In [1]:
%pwd

'd:\\Mini-Memento'

In [1]:
import os
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from pathlib import Path


DEFAULT_EXECUTE_TIMEOUT = 120

from loguru import logger
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
load_dotenv('.env')
from pathlib import Path

In [2]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend, FilesystemBackend
from langchain.chat_models import init_chat_model
# from langgraph.checkpoint.memory import InMemorySaver  # thêm dòng này
from deepagents.middleware.skills import SkillsMiddleware
from deepagents.middleware.filesystem import FilesystemMiddleware

In [3]:
sandbox_dir = Path("env/").absolute()
backend = LocalShellBackend(
    root_dir=sandbox_dir,
    virtual_mode=True,
    env={
        "PATH": os.environ["PATH"]
    },
    # inherit_env=True,
    timeout=120,
)

In [4]:
model = ChatOpenAI(
    model="gpt-4.1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# checkpointer = InMemorySaver() 

middleware = [
    SkillsMiddleware(backend=backend, sources=["/skills/"]),
    FilesystemMiddleware(backend=backend)
]

agent = create_agent(
    model=model,
    # checkpointer=checkpointer,
    middleware=middleware
)

In [5]:
user_prompt = 'What are project of this pdf "C:\\Users\\thang\\Downloads\\AI_Engineer_ThangCN.pdf"'
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_prompt,
            }
        ]
    },
    config={"configurable": {"thread_id": "12345"}},
)

In [6]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What are project of this pdf "C:\Users\thang\Downloads\AI_Engineer_ThangCN.pdf"
================================== Ai Message ==================================
Tool Calls:
  read_file (call_1zrTLFGIxs7dj862p3JWUrUB)
 Call ID: call_1zrTLFGIxs7dj862p3JWUrUB
  Args:
    file_path: C:\Users\thang\Downloads\AI_Engineer_ThangCN.pdf
================================= Tool Message =================================
Name: read_file

Error: Windows absolute paths are not supported: C:\Users\thang\Downloads\AI_Engineer_ThangCN.pdf. Please use virtual paths starting with / (e.g., /workspace/file.txt)
================================== Ai Message ==================================

It looks like you provided a local Windows file path (C:\Users\thang\Downloads\AI_Engineer_ThangCN.pdf). I don't have direct access to your local files. 

To help you review the projects from this PDF, please upload the PDF file here, or mov

In [11]:
from agents.define_agent import convert_result_agent, write_agent

result_convert = convert_result_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Convert the following skill execution result into a concise summary: " + str(result)
            }
        ]

    }
)
r_dict = result_convert['structured_response'].model_dump()
r_dict

{'skill': {'name': 'web-search',
  'created': False,
  'reason': 'The user asked to extract project information from a PDF file; accessing and reading files is associated with skills that handle external content, such as web-search or document-analysis skills.'},
 'trace': [{'step': 1,
   'action': 'route_skill',
   'detail': 'Skill selected based on the intent to extract information from a document.',
   'evidence': 'User requested project info from a PDF file.',
   'error': None},
  {'step': 2,
   'action': 'execute',
   'detail': 'Attempted to read a file using a provided Windows path.',
   'evidence': "read_file(file_path='C:\\Users\\thang\\Downloads\\AI_Engineer_ThangCN.pdf')",
   'error': {'type': 'UnsupportedPathError',
    'message': 'Windows absolute paths are not supported; only virtual paths are accepted.',
    'raw': 'Error: Windows absolute paths are not supported: C:\\Users\\thang\\Downloads\\AI_Engineer_ThangCN.pdf. Please use virtual paths...'}},
  {'step': 3,
   'actio

In [12]:
reward = r_dict.get("reward", {})
verdict = reward.get("verdict", "")
next_step = None
for v, b in verdict.items():
    if b:
        next_step = v
print("Next step:", next_step)

Next step: improve_existing_skill


In [15]:
next_step = "create_new_skill"

In [16]:
w_res = write_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"""Based on 
                user prompt: {user_prompt}
                and the skill execution result {r_dict}, the next step is: {next_step}. Please update skill or write a new skill to execute the next step."""
            }
        ]
    }
)

In [17]:
for m in w_res["messages"]:
    m.pretty_print()

================================ Human Message =================================

Based on 
                user prompt: What are project of this pdf "C:\Users\thang\Downloads\AI_Engineer_ThangCN.pdf"
                and the skill execution result {'skill': {'name': 'web-search', 'created': False, 'reason': 'The user asked to extract project information from a PDF file; accessing and reading files is associated with skills that handle external content, such as web-search or document-analysis skills.'}, 'trace': [{'step': 1, 'action': 'route_skill', 'detail': 'Skill selected based on the intent to extract information from a document.', 'evidence': 'User requested project info from a PDF file.', 'error': None}, {'step': 2, 'action': 'execute', 'detail': 'Attempted to read a file using a provided Windows path.', 'evidence': "read_file(file_path='C:\\Users\\thang\\Downloads\\AI_Engineer_ThangCN.pdf')", 'error': {'type': 'UnsupportedPathError', 'message': 'Windows absolute paths are not sup